# v4 Notebook C - GCG (A3) on S1 & S5, then re-score through L0
Run AFTER A. ~4 h on a T4; resumable. Output: `v4_C_gcg.npz`. Final success is the L0-re-scored number, never the raw GCG loss (pre-mortem M8).

In [ ]:
# Cell 1 - setup (Kaggle T4; Internet ON; Secrets: HF_TOKEN, AGENT_API_KEY)
import os, sys, subprocess, glob
KAGGLE = os.path.isdir('/kaggle/working')
DEST = ('/kaggle/working' if KAGGLE else '/content') + '/vyuha_src'
if not os.path.isdir(f'{DEST}/.git'):
    subprocess.run(['git','clone','--depth','1','https://github.com/g25ait2149/vyuha.git',DEST], check=False)
else:
    subprocess.run(['git','-C',DEST,'pull','--ff-only'], check=False)
hits = glob.glob(DEST+'/**/vyuha/__init__.py', recursive=True)
assert hits, 'clone failed - turn Internet ON'
ROOT = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, ROOT); os.chdir(ROOT)
subprocess.run('pip -q install -U "bitsandbytes>=0.46.1" accelerate "transformers>=4.44" datasets scikit-learn sentencepiece "protobuf<5" huggingface_hub', shell=True)
tok = ''
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata; tok = userdata.get('HF_TOKEN')
except Exception as e:
    print('HF_TOKEN missing:', repr(e)[:100])
os.environ['HF_TOKEN'] = os.environ['HUGGINGFACE_HUB_TOKEN'] = tok or ''
from huggingface_hub import login
if tok: login(tok)
import torch; print('GPUs:', torch.cuda.device_count(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
OUT = ('/kaggle/working' if KAGGLE else '/content') + '/v4'; os.makedirs(OUT, exist_ok=True)
print('OUT =', OUT)

In [ ]:
# Cell 2 - GCG (A3) on the open-weight LM guards S1, S5 (per-seed, 100 eval seeds). Resumable.
import json, os, numpy as np, torch
from vyuha.normalize.normalize import normalize
from vyuha.cascade.guards import load_model, forced_prefix, first_ids, VOCAB, base_prompt, clip_user
from vyuha.cascade.gcg import gcg_lm, suffix_vocab, splice_user_suffix
EVAL = json.load(open(f'{OUT}/v4_data.json'))['eval_seeds'][:100]
V1 = lambda t: normalize(t, full=True, version=2)
OUTF=f'{OUT}/v4_C_gcg.npz'; store=dict(np.load(OUTF)) if os.path.exists(OUTF) else {}
def put(k,v): store[k]=np.asarray(v); np.savez(OUTF, **store)

def run_gcg(tag, mid, kind, steps=100, n_sfx=20):      # batched candidate eval; ~2-3 h/screen on a T4 (pre-mortem M9)
    if f'{tag}/gcg_text' in store: print('[resume]', tag); return
    tok, m = load_model(mid, False)
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()): pre = forced_prefix(tok, m, kind, [t for t in EVAL if len(t)<300][:5])
    u = sorted(first_ids(tok, VOCAB[kind][0])); s = sorted(first_ids(tok, VOCAB[kind][1]))
    cand = suffix_vocab(tok)
    finals, scores, suffixes = [], [], []
    for i, seed in enumerate(EVAL):
        uview = clip_user(tok, V1(seed))
        # suffix lives INSIDE the user message, exactly where it will sit when re-scored (splice is round-trip exact)
        head_ids, tail_ids, n_fill = splice_user_suffix(tok, base_prompt, uview, kind, pre, n_sfx)
        adv, loss = gcg_lm(tok, m, head_ids, tail_ids, u, s, cand, n_tokens=n_fill, steps=steps,
                           log=(print if i < 2 else None))
        suffix = tok.decode(adv)
        finals.append(seed + ' ' + suffix); suffixes.append(suffix); scores.append(loss)
        if i % 10 == 0: print(f'  {tag} gcg {i}/{len(EVAL)} search-loss {loss:.2f}', flush=True)
    put(f'{tag}/gcg_text', np.array(finals, dtype=object)); put(f'{tag}/gcg_loss', np.array(scores))
    put(f'{tag}/gcg_suffix', np.array(suffixes, dtype=object))
    m=tok=None; import gc; gc.collect(); torch.cuda.empty_cache()
run_gcg('S1_qwen06', 'Qwen/Qwen3Guard-Gen-0.6B', 'qwen')
run_gcg('S5_lg3_1b', 'meta-llama/Llama-Guard-3-1B', 'llamaguard')
print('GCG done. NOTE: final success is re-scored through L0->screen in the merge step, not from gcg_loss.')

In [ ]:
# Cell 3 - re-score GCG finals through the REAL pipeline (L0 -> each screen). This is the number that counts (M8).
import json, numpy as np, torch
from vyuha.normalize.normalize import normalize
from vyuha.cascade.guards import load_model, forced_prefix, bucketed_scores
V = lambda xs: [normalize(t, full=True, version=2) for t in xs]
probes=[t for t in json.load(open(f'{OUT}/v4_data.json'))['eval_seeds'] if len(t)<300][:5]
for tag, mid, kind in [('S1_qwen06','Qwen/Qwen3Guard-Gen-0.6B','qwen'),('S5_lg3_1b','meta-llama/Llama-Guard-3-1B','llamaguard')]:
    if f'{tag}/gcg_text' not in store or f'{tag}/gcg_text_score' in store: continue
    tok,m=load_model(mid,False)
    import io,contextlib
    with contextlib.redirect_stdout(io.StringIO()): pre=forced_prefix(tok,m,kind,probes)
    finals=[str(x) for x in store[f'{tag}/gcg_text']]
    put(f'{tag}/gcg_text_score', bucketed_scores(tok,m,V(finals),kind,pre))
    m=tok=None; import gc; gc.collect(); torch.cuda.empty_cache()
print('re-scored GCG finals through L0 for:', [k for k in store if k.endswith("gcg_text_score")])